# Łańcuchy Markowa dla Pana Tadeusza (poziom słów)
W tym zeszycie budujemy model łańcuchów Markowa dla *Pana Tadeusza* używając słów zamiast pojedynczych znaków.

In [ ]:
import random
import re
from collections import defaultdict, Counter
from typing import Optional


In [ ]:
# Wczytujemy tekst z pliku
with open('data/pan-tadeusz.txt', 'r', encoding='utf-8') as f:
    text: str = f.read()

print(f"Całkowita liczba znaków: {len(text)}")

In [ ]:
# Tokenizujemy tekst na słowa i znaki interpunkcyjne
# To wyrażenie regularne znajduje słowa lub pojedyncze znaki niebędące białymi znakami (interpunkcja)
tokens: list[str] = re.findall(r'\b\w+\b|[^\w\s]', text)

print(f"Całkowita liczba tokenów: {len(tokens)}")
print(f"Unikalne tokeny: {len(set(tokens))}")

In [ ]:
# Budujemy model łańcuchów Markowa
def zbuduj_model_markowa(tokeny: list[str], rozmiar_stanu: int = 1) -> dict[tuple[str, ...], Counter]:
    """
    Tworzy model Markowa na podstawie listy tokenów.
    
    :param tokeny: Lista słów/znaków interpunkcyjnych z tekstu.
    :param rozmiar_stanu: Liczba tokenów brana pod uwagę przy przewidywaniu następnego (n-gram).
    :return: Słownik mapujący stan (krotkę tokenów) na licznik kolejnych tokenów.
    """
    model: dict[tuple[str, ...], Counter] = defaultdict(Counter)
    for i in range(len(tokeny) - rozmiar_stanu):
        stan: tuple[str, ...] = tuple(tokeny[i:i+rozmiar_stanu])
        nastepne_slowo: str = tokeny[i+rozmiar_stanu]
        model[stan][nastepne_slowo] += 1
    return model

rozmiar_stanu: int = 1
model: dict[tuple[str, ...], Counter] = zbuduj_model_markowa(tokens, rozmiar_stanu=rozmiar_stanu)
print(f"Liczba unikalnych stanów: {len(model)}")

In [ ]:
# Funkcja do generowania tekstu
def generuj_tekst(model: dict[tuple[str, ...], Counter], stan_poczatkowy: Optional[tuple[str, ...]] = None, dlugosc: int = 50) -> str:
    """
    Generuje tekst na podstawie zbudowanego modelu Markowa.
    
    :param model: Model Markowa (słownik stanów i liczników).
    :param stan_poczatkowy: Opcjonalny stan początkowy. Jeśli brak, losujemy.
    :param dlugosc: Liczba tokenów do wygenerowania.
    :return: Wygenerowany tekst jako string.
    """
    if stan_poczatkowy is None:
        stan_poczatkowy = random.choice(list(model.keys()))
    
    aktualny_stan: tuple[str, ...] = stan_poczatkowy
    wynik: list[str] = list(aktualny_stan)
    
    for _ in range(dlugosc):
        if aktualny_stan not in model:
            break
            
        wybory: list[str] = list(model[aktualny_stan].keys())
        wagi: list[int] = list(model[aktualny_stan].values())
        
        nastepny_token: str = random.choices(wybory, weights=wagi)[0]
        wynik.append(nastepny_token)
        
        # Aktualizujemy stan biorąc pod uwagę ostatnie 'n' tokenów
        aktualny_stan = tuple(wynik[-len(aktualny_stan):])
        
    # Prosta detokenizacja dla lepszego wyświetlania
    output: str = ""
    for token in wynik:
        if re.match(r'[^\w\s]', token):
            output += token
        else:
            output += " " + token
            
    return output.strip()


In [ ]:
# Generujemy trochę tekstu
wygenerowany: str = generuj_tekst(model, dlugosc=100)
print("Wygenerowany tekst (stan = 1):")
print(wygenerowany)

In [ ]:
# Próbujemy z rozmiarem stanu = 2 (bigramy) dla większej spójności
model_bigram: dict[tuple[str, ...], Counter] = zbuduj_model_markowa(tokens, rozmiar_stanu=2)
print("\nWygenerowany tekst (stan = 2):")
print(generuj_tekst(model_bigram, dlugosc=100))